# ДЗ 1

## Бэггинг и бустинг
Для датасета creditcardfraud (мошенничество с кредитными картами, 0 - нет мошенничества, 1 - есть)
- подобрать параметры с помощью GridSearchCV
- вывести и выбрать наилучшие
- обучить бэггинг и бустинг
- сравнить модели (вывести информацию по метрикам)

Для выполнения можно воспользоваться кодом с семинара. Код для загрузки датасета ниже:

In [2]:
from sklearn.model_selection import train_test_split, GridSearchCV
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier
from sklearn.base import clone
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix

In [3]:

df = pd.read_csv('./creditcard.csv')

In [4]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


# 1 - Analyzing the dataset.

In [5]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum().sum())

print("\nTarget counts:")
print(df["Class"].value_counts())
print("\nTarget proportions:")
print(df["Class"].value_counts(normalize=True))


Shape: (284807, 31)

Columns:
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

Missing values:
0

Target counts:
Class
0    284315
1       492
Name: count, dtype: int64

Target proportions:
Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64


# 2 - Training split and model setup

In [9]:
X = df.drop(columns="Class")
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training fraud cases:", y_train.sum())
print("Test fraud cases:", y_test.sum())

X_tune, _, y_tune, _ = train_test_split(
    X_train,
    y_train,
    train_size=0.1,
    stratify=y_train,
    random_state=42
)



tree = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

bagging = BaggingClassifier(
    estimator=tree,
    random_state=42,
    n_jobs=-1
)

boosting = AdaBoostClassifier(
    estimator=tree,
    random_state=42
)

Training fraud cases: 394
Test fraud cases: 98


# 3 - Choosing models with calculated params

In [12]:
bagging_grid = {
    "n_estimators": [50, 100],
    "max_samples": [0.7, 1.0],
    "max_features": [0.7, 1.0],
}

boosting_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.5, 1.0],
    "estimator__max_depth": [1, 2],
}

bagging_search = GridSearchCV(
    bagging,
    param_grid=bagging_grid,
    scoring="average_precision",
    cv=3,
    n_jobs=-1,
    verbose=0,
)

boosting_search = GridSearchCV(
    boosting,
    param_grid=boosting_grid,
    scoring="average_precision",
    cv=3,
    n_jobs=-1,
    verbose=0,
)

bagging_search.fit(X_tune, y_tune)
boosting_search.fit(X_tune, y_tune)

print("Bagging best parameters:", bagging_search.best_params_)
print("Bagging best CV average precision:", bagging_search.best_score_)

print("Boosting best parameters:", boosting_search.best_params_)
print("Boosting best CV average precision:", boosting_search.best_score_)

Bagging best parameters: {'max_features': 0.7, 'max_samples': 1.0, 'n_estimators': 50}
Bagging best CV average precision: 0.912176969904726
Boosting best parameters: {'estimator__max_depth': 2, 'learning_rate': 0.5, 'n_estimators': 50}
Boosting best CV average precision: 0.8790519886963947


# 4 - Training final models

In [ ]:
bagging_final = clone(bagging_search.best_estimator_)
boosting_final = clone(boosting_search.best_estimator_)

bagging_final.fit(X_train, y_train)
boosting_final.fit(X_train, y_train)

for name, model in [
    ("Bagging", bagging_final),
    ("Boosting", boosting_final),
]:
    y_pred = model.predict(X_test)
    y_scores = model.predict_proba(X_test)[:, 1]

    print(f"\n{name}")
    print("Average precision:", average_precision_score(y_test, y_scores))
    print("Precision:", precision_score(y_test, y_pred, zero_division=0))
    print("Recall:", recall_score(y_test, y_pred, zero_division=0))
    print("F1:", f1_score(y_test, y_pred, zero_division=0))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))